In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from google.colab import drive

In [ ]:
# Mount Google Drive to access files
try:
    drive.mount('/content/drive')
except Exception as e:
    raise RuntimeError(f"Failed to mount Google Drive: {e}")


Mounted at /content/drive


In [ ]:
# Load the CSV file from your Drive
data_path = '/content/drive/MyDrive/Final Year Project/merged_data.csv'

try:
    df = pd.read_csv(data_path)
    print(f"Initial dataset shape: {df.shape}")
except FileNotFoundError:
    raise FileNotFoundError(f"Dataset not found at {data_path}")
except Exception as e:
    raise RuntimeError(f"Error loading dataset: {e}")

# Check if it's empty
if df.empty:
    raise ValueError("The dataset is empty.")


Initial dataset shape: (105051, 7)


In [ ]:
# Fill missing values
df['sender'] = df['sender'].fillna('Unknown')
df['receiver'] = df['receiver'].fillna('Unknown')
df['date'] = df['date'].fillna('Unknown')
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')

# Remove duplicates
initial_rows = df.shape[0]
df = df.drop_duplicates()
print(f"Removed {initial_rows - df.shape[0]} duplicates. New shape: {df.shape}")

# Check for any remaining nulls
print("Null check:\n", df.isnull().sum())

# Create a new column 'full_text'
df['full_text'] = df['subject'] + ' ' + df['body']
texts = df['full_text'].values


Removed 1800 duplicates. New shape: (103251, 7)
Null check:
 sender      0
receiver    0
date        0
subject     0
body        0
label       0
urls        0
dtype: int64


In [ ]:
# Initialize the tokenizer and model
model_name = 'microsoft/deberta-v3-base'

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
except Exception as e:
    raise RuntimeError(f"Error loading DeBERTa model/tokenizer: {e}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()


In [ ]:
def get_embeddings(texts, tokenizer, model, device, batch_size=32):
    """Extract DeBERTa embeddings for a list of texts."""
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts.tolist(),
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)


In [ ]:
# Generate embeddings using DeBERTa
embeddings = get_embeddings(texts, tokenizer, model, device, batch_size=32)


Generating embeddings:   0%|          | 0/3227 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Generating embeddings: 100%|██████████| 3227/3227 [1:21:33<00:00,  1.52s/it]


In [31]:
# Save embeddings and tokenizer
output_dir = '/content/drive/MyDrive/Final Year Project'
output_path = f'{output_dir}/deberta_embeddings.npy'

try:
    np.save(output_path, embeddings)
    tokenizer.save_pretrained(f'{output_dir}/deberta_tokenizer')
    print(f'Embeddings saved to {output_path}. Shape: {embeddings.shape}')
    print('Tokenizer saved.')
except Exception as e:
    raise RuntimeError(f"Error saving outputs: {e}")

RuntimeError: Error saving outputs: name 'embeddings' is not defined

In [32]:
# Load CSV and NumPy embeddings
embeddings_path = '/content/drive/MyDrive/Final Year Project/deberta_embeddings.npy'

try:
    X = np.load(embeddings_path)
    print(f"Dataset shape: {df.shape}, Embeddings shape: {X.shape}")
except FileNotFoundError as e:
    raise FileNotFoundError(f"File not found: {e}")
except Exception as e:
    raise RuntimeError(f"Error loading data: {e}")

Dataset shape: (103251, 8), Embeddings shape: (103251, 768)


In [ ]:
# Ensure embedding count matches dataset rows
if X.shape[0] != df.shape[0]:
    raise ValueError(f"Embedding rows ({X.shape[0]}) don't match dataset rows ({df.shape[0]})")

# Extract labels
y = df['label'].values
print(f"Label distribution: Phishing (1): {np.sum(y == 1)}, Legitimate (0): {np.sum(y == 0)}")


Label distribution: Phishing (1): 56291, Legitimate (0): 46960


In [ ]:
import os
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score
from xgboost import XGBClassifier


In [ ]:
# Class weight to handle label imbalance (~53.6% phishing)
class_weight = {0: 1.0, 1: len(y) / (2 * np.sum(y == 1))}

# Initialize ML models
models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight=class_weight),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, scale_pos_weight=len(y) / np.sum(y == 1)),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42, class_weight=class_weight),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=100, random_state=42, class_weight=class_weight)
}


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score

# 5-Fold Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {name: {'f1': [], 'precision': [], 'accuracy': [], 'recall': []} for name in models}

# Output directory for saving trained models
output_dir = '/content/drive/MyDrive/Final Year Project/models'
os.makedirs(output_dir, exist_ok=True)

# Standard scaler for Logistic Regression
scaler = StandardScaler()


In [ ]:
from sklearn.metrics import accuracy_score, recall_score
import joblib

# 5-fold Stratified K-Fold Training with Progress Tracking
print("\n===== 5-Fold Stratified K-Fold Training =====")
for fold, (train_idx, val_idx) in enumerate(tqdm(skf.split(X, y), total=5, desc="Folds", unit="fold"), 1):
    print(f"\nFold {fold}/5")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Apply scaling for Logistic Regression
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Train each model with progress bar
    for name, model in tqdm(models.items(), desc=f"Fold {fold} Models", unit="model", leave=False):
        print(f"  Training {name}...")
        X_train_model = X_train_scaled if name == 'LogisticRegression' else X_train
        X_val_model = X_val_scaled if name == 'LogisticRegression' else X_val

        model.fit(X_train_model, y_train)
        y_pred = model.predict(X_val_model)

        f1 = f1_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        accuracy = accuracy_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)
        results[name]['f1'].append(f1)
        results[name]['precision'].append(precision)
        results[name]['accuracy'].append(accuracy)
        results[name]['recall'].append(recall)
        print(f"  {name:<20} F1: {f1:.4f}  Precision: {precision:.4f}  Accuracy: {accuracy:.4f}  Recall: {recall:.4f}")

        # Save model for this fold
        joblib.dump(model, f"{output_dir}/{name}_fold_{fold}.pkl")
    print("-" * 60)


===== 5-Fold Stratified K-Fold Training =====


Folds:   0%|          | 0/5 [00:00<?, ?fold/s]


Fold 1/5



Fold 1 Models:   0%|          | 0/5 [00:00<?, ?model/s]

  Training RandomForest...
  RandomForest         F1: 0.9781  Precision: 0.9765  Accuracy: 0.9761  Recall: 0.9797


...

  LogisticRegression   F1: 0.9877  Precision: 0.9881  Accuracy: 0.9866  Recall: 0.9874
  Training GradientBoosting...



Fold 5 Models:  80%|████████  | 4/5 [1:33:09<33:41, 2021.40s/model]

  GradientBoosting     F1: 0.9692  Precision: 0.9686  Accuracy: 0.9664  Recall: 0.9699
  Training ExtraTrees...
  ExtraTrees           F1: 0.9774  Precision: 0.9787  Accuracy: 0.9754  Recall: 0.9761



Folds: 100%|██████████| 5/5 [7:54:52<00:00, 5698.51s/fold]

------------------------------------------------------------


In [ ]:
# Calculate average and std metrics for each model
metrics = []
for name in models:
    avg_f1 = np.mean(results[name]['f1'])
    avg_precision = np.mean(results[name]['precision'])
    avg_accuracy = np.mean(results[name]['accuracy'])
    avg_recall = np.mean(results[name]['recall'])
    std_f1 = np.std(results[name]['f1'])
    std_precision = np.std(results[name]['precision'])
    std_accuracy = np.std(results[name]['accuracy'])
    std_recall = np.std(results[name]['recall'])

    metrics.append({
        'Model': name,
        'Avg_F1_Score': avg_f1,
        'Avg_Precision': avg_precision,
        'Avg_Accuracy': avg_accuracy,
        'Avg_Recall': avg_recall,
        'Std_F1_Score': std_f1,
        'Std_Precision': std_precision,
        'Std_Accuracy': std_accuracy,
        'Std_Recall': std_recall
    })

In [ ]:
# Save evaluation metrics
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(f'{output_dir}/model_metrics.csv', index=False)

# Print summary
print("\n📊 Average Metrics Across 5 Folds:")
print(metrics_df)


📊 Average Metrics Across 5 Folds:
                Model  Avg_F1_Score  Avg_Precision  Avg_Accuracy  Avg_Recall  \
0        RandomForest      0.978129       0.976607      0.976116    0.979659   
1             XGBoost      0.988055       0.985680      0.986944    0.990443   
2  LogisticRegression      0.988554       0.988230      0.987516    0.988879   
3    GradientBoosting      0.968807       0.966798      0.965918    0.970830   
4          ExtraTrees      0.977921       0.977783      0.975923    0.978060   

   Std_F1_Score  Std_Precision  Std_Accuracy  Std_Recall  
0      0.000826       0.000629      0.000879    0.001933  
1      0.000706       0.000968      0.000773    0.000834  
2      0.000755       0.000441      0.000818    0.001280  
3      0.001493       0.001837      0.001619    0.002483  
4      0.000958       0.000817      0.001031    0.001715  


In [ ]:
import joblib
from sklearn.ensemble import VotingClassifier

In [ ]:
model_dir = '//content/drive/MyDrive/Final Year Project/models'
model_names = ['RandomForest', 'XGBoost', 'LogisticRegression', 'GradientBoosting', 'ExtraTrees']
models = {}

for name in model_names:
    for fold in range(1, 6):
        model_path = f"{model_dir}/{name}_fold_{fold}.pkl"
        try:
            models[f"{name}_fold_{fold}"] = joblib.load(model_path)
            print(f"Loaded {name}_fold_{fold}")
        except FileNotFoundError:
            raise FileNotFoundError(f"Model not found: {model_path}")

print(f"Total models loaded: {len(models)}")

Loaded RandomForest_fold_1
Loaded RandomForest_fold_2
Loaded RandomForest_fold_3
Loaded RandomForest_fold_4
Loaded RandomForest_fold_5
Loaded XGBoost_fold_1
Loaded XGBoost_fold_2
Loaded XGBoost_fold_3
Loaded XGBoost_fold_4
Loaded XGBoost_fold_5
Loaded LogisticRegression_fold_1
Loaded LogisticRegression_fold_2
Loaded LogisticRegression_fold_3
Loaded LogisticRegression_fold_4
Loaded LogisticRegression_fold_5
Loaded GradientBoosting_fold_1
Loaded GradientBoosting_fold_2
Loaded GradientBoosting_fold_3
Loaded GradientBoosting_fold_4
Loaded GradientBoosting_fold_5
Loaded ExtraTrees_fold_1
Loaded ExtraTrees_fold_2
Loaded ExtraTrees_fold_3
Loaded ExtraTrees_fold_4
Loaded ExtraTrees_fold_5
Total models loaded: 25


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler = StandardScaler()

ensemble_probs = np.zeros(len(y))
ensemble_preds = np.zeros(len(y))
confidence_flags = np.zeros(len(y), dtype=int)
fold_metrics = {'f1': [], 'precision': [], 'accuracy': [], 'recall': []}

print("\n===== Ensemble Prediction ===")
for fold, (train_idx, val_idx) in enumerate(tqdm(skf.split(X, y), total=5, desc="Folds", unit="fold")):
    print(f"\nFold {fold + 1}/5")
    X_val = X[val_idx]
    y_val = y[val_idx]

    X_val_scaled = scaler.fit_transform(X_val)

    fold_probs = np.zeros((len(val_idx), len(models)))
    for i, (name, model) in enumerate(tqdm(models.items(), desc=f"Fold {fold + 1} Models", unit="model", leave=False)):
        X_val_pred = X_val_scaled if 'LogisticRegression' in name else X_val
        fold_probs[:, i] = model.predict_proba(X_val_pred)[:, 1]

    avg_probs = np.mean(fold_probs, axis=1)
    ensemble_probs[val_idx] = avg_probs
    ensemble_preds[val_idx] = (avg_probs >= 0.5).astype(int)

    confidence_flags[val_idx] = ((avg_probs >= 0.4) & (avg_probs <= 0.6)).astype(int)

    f1 = f1_score(y_val, ensemble_preds[val_idx])
    precision = precision_score(y_val, ensemble_preds[val_idx])
    accuracy = accuracy_score(y_val, ensemble_preds[val_idx])
    recall = recall_score(y_val, ensemble_preds[val_idx])

    fold_metrics['f1'].append(f1)
    fold_metrics['precision'].append(precision)
    fold_metrics['accuracy'].append(accuracy)
    fold_metrics['recall'].append(recall)

    print(f"  Ensemble Fold {fold + 1:<10} F1: {f1:.4f}  Precision: {precision:.4f}  Accuracy: {accuracy:.4f}  Recall: {recall:.4f}")
    print("-" * 60)



===== Ensemble Prediction ===


Folds:   0%|          | 0/5 [00:00<?, ?fold/s]


Fold 1/5



Folds:  20%|██        | 1/5 [00:10<00:43, 10.87s/fold]

  Ensemble Fold 1          F1: 0.9984  Precision: 0.9977  Accuracy: 0.9983  Recall: 0.9991
------------------------------------------------------------

Fold 2/5


...

  Ensemble Fold 3          F1: 0.9988  Precision: 0.9980  Accuracy: 0.9987  Recall: 0.9996
------------------------------------------------------------

Fold 4/5



Folds:  80%|████████  | 4/5 [00:44<00:11, 11.15s/fold]

  Ensemble Fold 4          F1: 0.9982  Precision: 0.9980  Accuracy: 0.9981  Recall: 0.9984
------------------------------------------------------------

Fold 5/5



Folds: 100%|██████████| 5/5 [00:55<00:00, 11.17s/fold]

  Ensemble Fold 5          F1: 0.9988  Precision: 0.9986  Accuracy: 0.9987  Recall: 0.9990
------------------------------------------------------------


In [ ]:
avg_metrics = {
    'Avg_F1_Score': np.mean(fold_metrics['f1']),
    'Avg_Precision': np.mean(fold_metrics['precision']),
    'Avg_Accuracy': np.mean(fold_metrics['accuracy']),
    'Avg_Recall': np.mean(fold_metrics['recall']),
    'Std_F1_Score': np.std(fold_metrics['f1']),
    'Std_Precision': np.std(fold_metrics['precision']),
    'Std_Accuracy': np.std(fold_metrics['accuracy']),
    'Std_Recall': np.std(fold_metrics['recall'])
}

metrics_df = pd.DataFrame([avg_metrics])
metrics_df.to_csv('/content/drive/My Drive/Final Year Project/ensemble_metrics.csv', index=False)
print("\n===== Average Ensemble Metrics =====")
print(metrics_df.to_string(index=False))
print("Metrics saved to ensemble_metrics.csv")



===== Average Ensemble Metrics =====
 Avg_F1_Score  Avg_Precision  Avg_Accuracy  Avg_Recall  Std_F1_Score  Std_Precision  Std_Accuracy  Std_Recall
     0.998526        0.99803      0.998392    0.999023      0.000234       0.000309      0.000255    0.000368
Metrics saved to ensemble_metrics.csv


In [ ]:
output_df = pd.DataFrame({
    'label': y,
    'ensemble_prediction': ensemble_preds,
    'ensemble_probability': ensemble_probs,
    'uncertain_flag': confidence_flags
})

output_df.to_csv('/content/drive/My Drive/Final Year Project/ensemble_predictions.csv', index=False)
print("Predictions saved to ensemble_predictions.csv")
print(f"Uncertain predictions (0.4–0.6): {np.sum(confidence_flags)} ({np.mean(confidence_flags)*100:.2f}% of total)")


Predictions saved to ensemble_predictions.csv
Uncertain predictions (0.4–0.6): 468 (0.45% of total)


In [ ]:
estimators = [(name, model) for name, model in models.items()]
ensemble = VotingClassifier(estimators=estimators, voting='soft')


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler = StandardScaler()

ensemble_probs = np.zeros(len(y))
ensemble_preds = np.zeros(len(y))
confidence_flags = np.zeros(len(y), dtype=int)
fold_metrics = {'f1': [], 'precision': [], 'accuracy': [], 'recall': []}


In [ ]:
print("\n===== Ensemble Prediction and Confidence Tracking =====")
for fold, (train_idx, val_idx) in enumerate(tqdm(skf.split(X, y), total=5, desc="Folds", unit="fold")):
    print(f"\nFold {fold + 1}/5")
    X_val = X[val_idx]
    y_val = y[val_idx]

    X_val_scaled = scaler.fit_transform(X_val)

    fold_probs = np.zeros((len(val_idx), len(models)))
    for i, (name, model) in enumerate(tqdm(models.items(), desc=f"Fold {fold + 1} Models", unit="model", leave=False)):
        X_val_model = X_val_scaled if 'LogisticRegression' in name else X_val
        fold_probs[:, i] = model.predict_proba(X_val_model)[:, 1]

    avg_probs = np.mean(fold_probs, axis=1)
    ensemble_probs[val_idx] = avg_probs
    ensemble_preds[val_idx] = (avg_probs >= 0.5).astype(int)

    confidence_flags[val_idx] = ((avg_probs >= 0.4) & (avg_probs <= 0.6)).astype(int)

    f1 = f1_score(y_val, ensemble_preds[val_idx])
    precision = precision_score(y_val, ensemble_preds[val_idx])
    accuracy = accuracy_score(y_val, ensemble_preds[val_idx])
    recall = recall_score(y_val, ensemble_preds[val_idx])
    fold_metrics['f1'].append(f1)
    fold_metrics['precision'].append(precision)
    fold_metrics['accuracy'].append(accuracy)
    fold_metrics['recall'].append(recall)
    print(f"  Ensemble Fold {fold + 1:<10} F1: {f1:.4f}  Precision: {precision:.4f}  Accuracy: {accuracy:.4f}  Recall: {recall:.4f}")
    print("-" * 60)



===== Ensemble Prediction and Confidence Tracking =====


Folds:   0%|          | 0/5 [00:00<?, ?fold/s]


Fold 1/5



Folds:  20%|██        | 1/5 [00:13<00:55, 13.84s/fold]

  Ensemble Fold 1          F1: 0.9984  Precision: 0.9977  Accuracy: 0.9983  Recall: 0.9991
------------------------------------------------------------

Fold 2/5


...

  Ensemble Fold 3          F1: 0.9988  Precision: 0.9980  Accuracy: 0.9987  Recall: 0.9996
------------------------------------------------------------

Fold 4/5



Folds:  80%|████████  | 4/5 [00:57<00:13, 13.80s/fold]

  Ensemble Fold 4          F1: 0.9982  Precision: 0.9980  Accuracy: 0.9981  Recall: 0.9984
------------------------------------------------------------

Fold 5/5



Folds: 100%|██████████| 5/5 [01:12<00:00, 14.52s/fold]

  Ensemble Fold 5          F1: 0.9988  Precision: 0.9986  Accuracy: 0.9987  Recall: 0.9990
------------------------------------------------------------


In [ ]:
ensemble_path = '/content/drive/MyDrive/Final Year Project/ensemble_model.pkl'
joblib.dump(ensemble, ensemble_path)
print(f"Ensemble model saved to {ensemble_path}")


Ensemble model saved to /content/drive/MyDrive/Final Year Project/ensemble_model.pkl


In [ ]:
avg_metrics = {
    'Avg_F1_Score': np.mean(fold_metrics['f1']),
    'Avg_Precision': np.mean(fold_metrics['precision']),
    'Avg_Accuracy': np.mean(fold_metrics['accuracy']),
    'Avg_Recall': np.mean(fold_metrics['recall']),
    'Std_F1_Score': np.std(fold_metrics['f1']),
    'Std_Precision': np.std(fold_metrics['precision']),
    'Std_Accuracy': np.std(fold_metrics['accuracy']),
    'Std_Recall': np.std(fold_metrics['recall'])
}

metrics_df = pd.DataFrame([avg_metrics])
metrics_df.to_csv('/content/drive/MyDrive/Final Year Project/ensemble_metrics.csv', index=False)
print("\n===== Average Ensemble Metrics =====")
print(metrics_df.to_string(index=False))
print("Metrics saved to ensemble_metrics.csv")



===== Average Ensemble Metrics =====
 Avg_F1_Score  Avg_Precision  Avg_Accuracy  Avg_Recall  Std_F1_Score  Std_Precision  Std_Accuracy  Std_Recall
     0.998526        0.99803      0.998392    0.999023      0.000234       0.000309      0.000255    0.000368
Metrics saved to ensemble_metrics.csv


In [ ]:
output_df = pd.DataFrame({
    'label': y,
    'ensemble_prediction': ensemble_preds,
    'ensemble_probability': ensemble_probs,
    'uncertain_flag': confidence_flags
})
output_df.to_csv('/content/drive/MyDrive/Final Year Project/ensemble_predictions.csv', index=False)
print("Predictions saved to ensemble_predictions.csv")
print(f"Uncertain predictions (0.4–0.6): {np.sum(confidence_flags)} ({np.mean(confidence_flags)*100:.2f}% of total)")


Predictions saved to ensemble_predictions.csv
Uncertain predictions (0.4–0.6): 468 (0.45% of total)


In [33]:

# ===== TEST ENSEMBLE MODEL ON 20 SAMPLE EMAILS =====

# Mount Google Drive
from google.colab import drive
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, accuracy_score, recall_score
import torch # Import torch for device handling
from transformers import AutoTokenizer, AutoModel # Import necessary transformers components

try:
    drive.mount('/content/drive')
except Exception as e:
    raise RuntimeError(f"Failed to mount Google Drive: {e}")

# Define 20 sample emails
phishing_samples = [
    "Urgent! Your account is locked. Click to unlock: http://secure-login-fake.com",
    "YOU WON $1,000,000! Send bank details to claim prize!",
    "Dear customer, your Amazon account is suspended. Verify now: http://amazon-verify.net",
    "Your IRS refund is ready! Claim here: http://irs-claim-online.org",
    "Warning: Suspicious login detected. Reset password: http://account-reset-fake.com",
    "Google Security: Unusual activity detected. Review here: http://google-account-check.com",
    "HR Update: Your payroll details need verification. Access portal: http://payroll-secure.info",
    "Microsoft Alert: Confirm your account ownership: http://ms-account-recovery.net",
    "IT Dept: Mandatory password update required. Click: http://it-policy-update.org",
    "Please review the attached invoice for payment. [Attachment: invoice.exe]"
]

legit_samples = [
    "Team meeting scheduled for Monday, 10 AM, Conference Room B. Agenda attached.",
    "Your order #584920 has shipped. Track at: https://official-shipping.com/track",
    "Subscription renewal notice: Your Netflix plan renews on June 10, 2025.",
    "Thank you for your payment of $49.99. Receipt available in your account.",
    "Weekly team update: Project milestones due by Friday. Details in shared drive.",
    "GitHub: Password changed on June 5, 2025. If not you, review: https://github.com/security",
    "Dropbox: New login from Lagos detected. Ignore if this was you.",
    "Bank of America: April 2025 statement ready. [PDF attached]",
    "Delivery Attempt: Package missed. Reschedule: https://ups.com/reschedule",
    "Microsoft: New sign-in from Nigeria on June 5, 2025. No action needed if authorized."
]

# Combine emails and labels
emails = phishing_samples + legit_samples
labels = [1] * 10 + [0] * 10  # 1 = phishing, 0 = legitimate
test_df = pd.DataFrame({'email': emails, 'label': labels})

# Re-initialize tokenizer and model for testing (or ensure they are available from previous cells)
# Assuming model_name, tokenizer, model, and device are available from previous cells
try:
    # If tokenizer and model are not in the global scope, load them here
    model_name = 'microsoft/deberta-v3-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    print("✅ Loaded DeBERTa model and tokenizer for testing.")
except NameError:
    print("DeBERTa model, tokenizer, or device not found in global scope. Loading...")
    # Load them if they are not defined globally
    model_name = 'microsoft/deberta-v3-base'
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
    except Exception as e:
        raise RuntimeError(f"Error loading DeBERTa model/tokenizer: {e}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    print("✅ Successfully loaded DeBERTa model and tokenizer for testing.")
except Exception as e:
     raise RuntimeError(f"An unexpected error occurred while loading model/tokenizer: {e}")


# Ensure the get_embeddings function is defined (copy-pasted from the original code)
def get_embeddings(texts, tokenizer, model, device, batch_size=32):
    """Extract DeBERTa embeddings for a list of texts."""
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i + batch_size]
        # Convert batch_texts to a list of strings if it's a pandas Series or similar
        if not isinstance(batch_texts, list):
            batch_texts = batch_texts.tolist()

        inputs = tokenizer(
            batch_texts, # Pass the list of strings
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

# Generate embeddings using DeBERTa - Corrected function name and arguments
X_test = get_embeddings(test_df['email'].tolist(), tokenizer, model, device)


# Load ensemble model
ensemble_path = '/content/drive/MyDrive/Final Year Project/ensemble_model.pkl'
try:
    # Ensure 'ensemble' variable is not overwritten later
    # If 'ensemble' was overwritten by the VotingClassifier object, joblib.load should work.
    # If it was somehow lost, reload it here.
    if 'ensemble' not in globals() or not isinstance(ensemble, VotingClassifier):
        print("Ensemble model not found in global scope. Loading...")
        ensemble = joblib.load(ensemble_path)
    print(f"✅ Loaded ensemble model from {ensemble_path}")
except FileNotFoundError:
    raise FileNotFoundError(f"❌ Ensemble model not found: {ensemble_path}")
except Exception as e:
    raise RuntimeError(f"Error loading ensemble model: {e}")


# Scale features (for models that need it like Logistic Regression)
# Use the same scaler used during training if possible, otherwise fit_transform on test data.
# Assuming the original scaler `scaler` is available and fitted from training data.
# If not, fit a new one on the test data.
try:
    X_test_scaled = scaler.transform(X_test) # Use transform if scaler was fitted on training
except NameError:
    print("Scaler not found in global scope. Fitting new scaler on test data.")
    scaler = StandardScaler()
    X_test_scaled = scaler.fit_transform(X_test)
except Exception as e:
    print(f"Error transforming test data with scaler: {e}. Fitting a new scaler.")
    scaler = StandardScaler()
    X_test_scaled = scaler.fit_transform(X_test)


# Predict with ensemble model (adjust if using manual averaging instead)
# Need to handle scaling correctly within the ensemble prediction if models require it.
# The saved `VotingClassifier` does not automatically handle individual model scaling.
# If the saved ensemble is a `VotingClassifier`, it expects the input format that works for its estimators.
# Since the individual models were trained *after* scaling for Logistic Regression,
# but *before* scaling for others, predicting with the VotingClassifier directly
# after just scaling the input `X_test` might be incorrect depending on how
# the individual models inside the VotingClassifier were pickled.

# A safer approach for ensemble prediction if individual models need different scaling:
# Predict probabilities for each individual model and average them, similar to the cross-validation loop.
# This requires loading *all* the individual models used in the original ensemble prediction loop (5 folds * 5 models = 25 models).
# The current code loads the single `VotingClassifier` object saved *after* cross-validation,
# which might not contain all the necessary individual models from all folds or handle the scaling correctly.

# Let's try predicting with the loaded VotingClassifier first, assuming it was saved correctly.
# If it still fails or gives bad results, the alternative is to manually predict and average as done in the CV loop.

try:
    # The VotingClassifier expects input that works for all its estimators.
    # Since LogisticRegression needed scaling, the VotingClassifier might expect scaled input,
    # or its internal estimators somehow handle it (less likely for simple VotingClassifier).
    # Let's pass the scaled data first, as LogisticRegression needs it.
    # If other models inside the ensemble were NOT trained on scaled data, this will cause issues.

    # **Revised Strategy:** Manually predict with each model from each fold and average,
    # applying scaling only where needed, just like in the cross-validation loop.

    # Load all individual models used in the ensemble prediction loop
    model_dir = '/content/drive/MyDrive/Final Year Project/models'
    model_names = ['RandomForest', 'XGBoost', 'LogisticRegression', 'GradientBoosting', 'ExtraTrees']
    loaded_individual_models = {}
    for name in model_names:
        loaded_individual_models[name] = []
        for fold in range(1, 6):
            model_path = f"{model_dir}/{name}_fold_{fold}.pkl"
            try:
                loaded_individual_models[name].append(joblib.load(model_path))
            except FileNotFoundError:
                 raise FileNotFoundError(f"Individual model not found: {model_path}. Cannot perform manual ensemble prediction.")

    # Perform manual ensemble prediction on X_test
    num_samples = len(X_test)
    total_folds = 5 # Assuming 5 folds were used for training the models saved
    total_models_per_sample = total_folds * len(model_names) # 5 folds * 5 models = 25 predictions per sample

    all_fold_model_probs = np.zeros((num_samples, total_models_per_sample))

    # Since we don't have train/val splits for the test data itself, we'll apply
    # each trained fold model to the entire test set X_test.
    # We still need to handle the scaler appropriately.
    # We will fit the scaler ONCE on X_test for models that need scaled data.
    X_test_scaled_manual = scaler.fit_transform(X_test)


    model_index = 0
    # Iterate through each model type and each fold's trained model
    print("\nGenerating predictions from individual fold models...")
    for name in tqdm(model_names, desc="Model Types", unit="type"):
        for fold_model in tqdm(loaded_individual_models[name], desc=f"  {name} Folds", unit="fold", leave=False):
            # Apply scaling only for Logistic Regression models
            X_test_pred = X_test_scaled_manual if name == 'LogisticRegression' else X_test
            all_fold_model_probs[:, model_index] = fold_model.predict_proba(X_test_pred)[:, 1]
            model_index += 1

    # Average the probabilities across all loaded fold models for each sample
    avg_probs = np.mean(all_fold_model_probs, axis=1)

    # Calculate ensemble predictions and confidence flags
    preds = (avg_probs >= 0.5).astype(int)
    confidence_flags = ((avg_probs >= 0.4) & (avg_probs <= 0.6)).astype(int)

except Exception as e:
    print(f"Error during manual ensemble prediction: {e}")
    # Fallback to predicting with the single saved VotingClassifier if manual fails
    print("Attempting prediction with the single saved VotingClassifier instead.")
    # Assuming the single saved VotingClassifier can handle the input correctly
    # Pass scaled data as LogisticRegression needs it, hoping other models handle it.
    try:
        probs = ensemble.predict_proba(X_test_scaled)[:, 1] # Use scaled data
        preds = (probs >= 0.5).astype(int)
        confidence_flags = ((probs >= 0.4) & (probs <= 0.6)).astype(int)
        avg_probs = probs # In this case, avg_probs is just the single ensemble output
        print("✅ Prediction with saved VotingClassifier successful.")
    except Exception as e_vc:
        raise RuntimeError(f"❌ Prediction with saved VotingClassifier also failed: {e_vc}")


# Evaluate
f1 = f1_score(test_df['label'], preds)
precision = precision_score(test_df['label'], preds)
accuracy = accuracy_score(test_df['label'], preds)
recall = recall_score(test_df['label'], preds)

# Display results
print("\n===== 📊 Ensemble Test Results on 20 Emails =====")
print(f"F1 Score     : {f1:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Accuracy     : {accuracy:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"Uncertain Predictions (0.4–0.6): {np.sum(confidence_flags)} ({np.mean(confidence_flags)*100:.2f}%)")

for i, row in test_df.iterrows():
    pred_label = 'Phishing' if preds[i] == 1 else 'Legitimate'
    actual_label = 'Phishing' if row['label'] == 1 else 'Legitimate'
    confidence = avg_probs[i] * 100 # Use avg_probs from either manual or VC prediction
    flag = 'Uncertain' if confidence_flags[i] == 1 else 'Confident'
    print(f"\n📧 Email {i+1}:")
    print(f"Text: {row['email'][:100]}...")
    print(f"Predicted: {pred_label} (Confidence: {confidence:.2f}%, {flag}) | Actual: {actual_label}")

# Save results
output_df = pd.DataFrame({
    'email': test_df['email'],
    'actual_label': test_df['label'],
    'ensemble_prediction': preds,
    'ensemble_probability': avg_probs, # Save the averaged probabilities
    'uncertain_flag': confidence_flags
})
output_path = '/content/drive/MyDrive/Final Year Project/test_20_emails_results.csv'
output_df.to_csv(output_path, index=False)
print(f"\n✅ Test results saved to: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Generating embeddings: 100%|██████████| 1/1 [00:03<00:00,  3.46s/it]


✅ Loaded ensemble model from /content/drive/MyDrive/Final Year Project/ensemble_model.pkl

Generating predictions from individual fold models...


  RandomForest Folds:   0%|          | 0/5 [00:00<?, ?fold/s]
                                                             
  XGBoost Folds:   0%|          | 0/5 [00:00<?, ?fold/s]
                                                        
  LogisticRegression Folds:   0%|          | 0/5 [00:00<?, ?fold/s]
                                                                   
  GradientBoosting Folds:   0%|          | 0/5 [00:00<?, ?fold/s]
                                                                 
Model Types: 100%|██████████| 5/5 [00:00<00:00, 39.28type/s]


===== 📊 Ensemble Test Results on 20 Emails =====
F1 Score     : 0.7200
Precision    : 0.6000
Accuracy     : 0.6500
Recall       : 0.9000
Uncertain Predictions (0.4–0.6): 4 (20.00%)

📧 Email 1:
Text: Urgent! Your account is locked. Click to unlock: http://secure-login-fake.com...
Predicted: Phishing (Confidence: 96.41%, Confident) | Actual: Phishing

📧 Email 2:
Text: YOU WON $1,000,000! Send bank details to claim prize!...
Predicted: Phishing (Confidence: 88.45%, Confident) | Actual: Phishing

📧 Email 3:
Text: Dear customer, your Amazon account is suspended. Verify now: http://amazon-verify.net...
Predicted: Phishing (Confidence: 93.81%, Confident) | Actual: Phishing

📧 Email 4:
Text: Your IRS refund is ready! Claim here: http://irs-claim-online.org...
Predicted: Phishing (Confidence: 77.92%, Confident) | Actual: Phishing

📧 Email 5:
Text: Warning: Suspicious login detected. Reset password: http://account-reset-fake.com...
Predicted: Phishing (Confidence: 93.58%, Confident) | Actual: P